# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", getattr(metadata, 'name', 'N/A'))
print("Description:", getattr(metadata, 'description', 'N/A'))

# If you want to inspect all available high-level attributes
for attr in dir(metadata):
    if not attr.startswith('_') and not callable(getattr(metadata, attr)):
        print(f"{attr}: {getattr(metadata, attr)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for reference in later steps.

We first inspect the available record sets, and then, for a chosen record set (by its `@id`), we inspect its fields (columns/attributes).

In [ ]:
# List available record sets and their IDs
record_sets = getattr(metadata, 'recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]
record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]

print("Available Record Sets (`@id`):")
for i, rs in enumerate(record_sets):
    if isinstance(rs, dict):
        print(f"  {i+1}. @id: {rs.get('@id')}, name: {rs.get('name', '<none>')}")
    else:
        print(f"  {i+1}. {rs}")
        
# If none are present, try extracting them from the metadata fileObjects
if not record_set_ids:
    print("No recordSet entries found in metadata.")
    print("Attempting to find record sets from available fileObjects...")
    if hasattr(metadata, 'fileObject'):
        file_objects = metadata.fileObject
        if not isinstance(file_objects, list):
            file_objects = [file_objects]
        for fobj in file_objects:
            print(f"FileObject @id: {fobj.get('@id')}, name: {fobj.get('name', '<none>')}, encodingFormat: {fobj.get('encodingFormat', '<none>')}")

### Example: Preview records for a record set
Let's try to preview a small sample from one of the primary data record sets (use the appropriate `@id` identified above). You may need to update the `record_set_id` variable if the dataset structure is updated.

In [ ]:
# Try to use a record set.
# If recordSet is empty (as in this package's metadata), attempt to obtain the actual records from the main data file
# Use the first file object as a record set

if hasattr(metadata, 'fileObject'):
    file_objects = metadata.fileObject
    if not isinstance(file_objects, list):
        file_objects = [file_objects]
    primary_fileobj = file_objects[0] if file_objects else None
    record_set_id = primary_fileobj['@id'] if primary_fileobj is not None else None
    print(f'Chosen record set (@id): {record_set_id}')
    
    print('Previewing first five records:')
    try:
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            print(record)
            if i >= 4:
                break
    except Exception as e:
        print("Could not preview records:", e)
else:
    print("No fileObject found in metadata. Cannot preview records.")

## 3. Data Extraction
Load data from the record set (`@id`) into a Pandas DataFrame for analysis.

Here, we build a DataFrame from the main record set. We also print the DataFrame's columns and a sample of the data. All references use the `@id` of the record set and fields.

In [ ]:
# Build a list of record set IDs to load (can have more than one in some datasets)
record_set_ids = [record_set_id] if 'record_set_id' in locals() and record_set_id is not None else []
dataframes = {}

for rsid in record_set_ids:
    print(f"Loading records from record set: {rsid}")
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

if record_set_ids:
    first_rsid = record_set_ids[0]
    print(f"Columns in DataFrame for record set {first_rsid}:")
    print(dataframes[first_rsid].columns.tolist())
    dataframes[first_rsid].head()
else:
    print("No record set IDs found; cannot extract data.")

## 4. Exploratory Data Analysis (EDA)
We demonstrate a few basic operations on the main record set:
- Filtering records based on a numeric field (e.g., by protein abundance or molecular weight)
- Normalizing a numeric field
- Grouping by a categorical field (e.g., protein accession or gene symbol if present)

**All fields are referenced by their `@id` (i.e., their DataFrame column name as loaded via Croissant).** Please adjust the field IDs as necessary for your specific analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ---
# STEP 1: Choose a numeric field by inspecting columns
if record_set_ids:
    df = dataframes[record_set_ids[0]]
    print("Columns in the main data:", df.columns.tolist())
    
    # Try a few likely candidate columns (edit the id below if another is more appropriate)
    candidate_numeric_fields = [col for col in df.columns if ('abundance' in col.lower() or 'mw' in col.lower() or 'weight' in col.lower() or col.lower().startswith('num') or 'count' in col.lower())]
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]
    else:
        # If no obvious field found, print columns for user to choose
        print('No numeric field with standard naming found, using first column.')
        numeric_field_id = df.columns[0]

    print(f"Using numeric field with @id: {numeric_field_id}")

    # Try to ensure numeric conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Example filter threshold
    threshold = np.nanmean(df[numeric_field_id])
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find a suitable group field (e.g. 'accession', 'description', gene symbol, or experimental condition)
    candidate_group_fields = [col for col in df.columns if any(k in col.lower() for k in ['accession','sample','type','condition','group','gene'])]
    if candidate_group_fields:
        group_field_id = candidate_group_fields[0]
        print(f"\nGrouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable group field found in columns.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
We demonstrate how to visualize data distributions of the filtered and normalized numeric field, as well as the grouped means per group field, using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids:
    df = dataframes[record_set_ids[0]]
    # Check if filtered_df and normalized column exist
    if 'filtered_df' in locals() and f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of normalized {numeric_field_id}")
        plt.xlabel(f"{numeric_field_id} (normalized)")
        plt.ylabel("Count")
        plt.show()

    # If grouped_df exists, show top 15 groups for comparison
    if 'grouped_df' in locals():
        plt.figure(figsize=(12,5))
        order = grouped_df[numeric_field_id].sort_values(ascending=False).index[:15]
        sns.barplot(data=grouped_df.head(15), x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 15)")
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process a Croissant FAIR<sup>2</sup> dataset (referenced by all `@id`s) describing mass spectrometry analysis of extracellular vesicle proteins.

- **Loading and schema exploration:** We showed how to access record sets and reference all schema entities using their `@id`.
- **Data extraction:** All records were loaded with fields and columns referenced by their `@id`.
- **EDA:** We filtered, normalized, and grouped data dynamically based on available numeric and categorical fields.
- **Visualization:** Common distribution and group-mean plots were demonstrated.

Further downstream analyses can be performed as appropriate (e.g., machine learning, feature selection, comparative expression) using the DataFrame(s) constructed above.